# Kaggle One-Cell — TSBD 30-case Bootstrap Verification
Add/upload one genuine 30-case CSV, then run the single code cell below. Required columns: `image_id,true_label,yolo_only_pred,yolo_ontology_pred`.


In [ ]:
# KAGGLE ONE-CELL: TSBD 30-case paired bootstrap verification
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score
B=2000;SEED=42;OUT=Path('/kaggle/working/tsbd30_bootstrap_output');OUT.mkdir(parents=True,exist_ok=True)
required=['image_id','true_label','yolo_only_pred','yolo_ontology_pred']
candidates=[]
for root in [Path('/kaggle/input'),Path('/kaggle/working')]:
    if root.exists():
        for p in root.rglob('*.csv'):
            try:
                h=pd.read_csv(p,nrows=3)
                if all(c in h.columns for c in required): candidates.append(p)
            except Exception: pass
if not candidates: raise FileNotFoundError('No compatible CSV found. Required columns: '+','.join(required))
if len(candidates)>1:
    print('Multiple compatible CSVs found:');[print(i+1,p) for i,p in enumerate(candidates)]
INPUT=candidates[0];print('Using:',INPUT);df=pd.read_csv(INPUT)
if len(df)!=30: raise ValueError(f'Expected 30 cases, found {len(df)}')
if df[required].isna().any().any(): raise ValueError('Blank/NA values found')
for c in required:
    if df[c].astype(str).str.strip().eq('').any(): raise ValueError(f'Blank values in {c}')
if df.image_id.duplicated().any(): raise ValueError('Duplicate image_id values')
for c in ['true_label','yolo_only_pred','yolo_ontology_pred']:
    if not pd.api.types.is_numeric_dtype(df[c]): df[c]=df[c].astype(str).str.strip()
labels=sorted(set(df.true_label)|set(df.yolo_only_pred)|set(df.yolo_ontology_pred),key=str)
y=df.true_label.to_numpy();a=df.yolo_only_pred.to_numpy();b=df.yolo_ontology_pred.to_numpy()
mf=lambda t,p:f1_score(t,p,labels=labels,average='macro',zero_division=0)
point_a,point_b=mf(y,a),mf(y,b);acc_a,acc_b=accuracy_score(y,a),accuracy_score(y,b)
rng=np.random.default_rng(SEED);ba=np.empty(B);bb=np.empty(B);dd=np.empty(B)
for i in range(B):
    idx=rng.integers(0,len(y),len(y));ba[i]=mf(y[idx],a[idx]);bb[i]=mf(y[idx],b[idx]);dd[i]=bb[i]-ba[i]
lo,hi=np.percentile(dd,[2.5,97.5]);p=min(1.0,2*min(np.mean(dd<=0),np.mean(dd>=0)))
summary=pd.DataFrame([{'input_file':str(INPUT),'n_cases':len(df),'labels':'|'.join(map(str,labels)),'bootstrap_resamples':B,'random_seed':SEED,'yolo_only_point_macro_f1':point_a,'yolo_ontology_point_macro_f1':point_b,'point_difference_ontology_minus_yolo':point_b-point_a,'yolo_only_accuracy':acc_a,'yolo_ontology_accuracy':acc_b,'bootstrap_mean_macro_f1_yolo_only':ba.mean(),'bootstrap_mean_macro_f1_yolo_ontology':bb.mean(),'bootstrap_mean_difference_ontology_minus_yolo':dd.mean(),'bootstrap_diff_ci95_low':lo,'bootstrap_diff_ci95_high':hi,'bootstrap_empirical_two_sided_p':p,'historical_yolo_ontology_bootstrap_mean_UNVERIFIED':0.556,'historical_yolo_only_bootstrap_mean_UNVERIFIED':0.564,'historical_mean_difference_UNVERIFIED':-0.008,'historical_p_value_UNVERIFIED':0.53}])
summary.to_csv(OUT/'bootstrap_30case_results.csv',index=False)
pd.DataFrame({'resample':np.arange(1,B+1),'macro_f1_yolo_only':ba,'macro_f1_yolo_ontology':bb,'diff_ontology_minus_yolo':dd}).to_csv(OUT/'bootstrap_30case_resamples.csv',index=False)
df.to_csv(OUT/'tsbd_30_predictions_VERIFIED_INPUT_COPY.csv',index=False)
with (OUT/'bootstrap_30case_results.txt').open('w',encoding='utf-8') as f:
    f.write('TSBD 30-CASE PAIRED BOOTSTRAP VERIFICATION\n'+'='*48+'\n\n')
    f.write(f'Input: {INPUT}\nN: {len(df)}\nLabels: {labels}\nBootstrap: {B}\nSeed: {SEED}\n\n')
    f.write(f'YOLO-only point Macro-F1: {point_a:.6f}\nYOLO+Ontology point Macro-F1: {point_b:.6f}\n')
    f.write(f'YOLO-only accuracy: {acc_a:.6f}\nYOLO+Ontology accuracy: {acc_b:.6f}\n\n')
    f.write(f'Bootstrap mean YOLO-only: {ba.mean():.6f}\nBootstrap mean YOLO+Ontology: {bb.mean():.6f}\n')
    f.write(f'Mean paired difference: {dd.mean():.6f}\n95% percentile CI: [{lo:.6f}, {hi:.6f}]\nTwo-sided empirical sign p: {p:.6f}\n')
print('\n=== RESULTS ===');display(summary.T)
print('\n=== FILES WRITTEN ===');[print(x) for x in OUT.iterdir()]
print('\nHistorical comparison (UNVERIFIED): Ontology=0.556; YOLO=0.564; diff=-0.008; p=0.53')
print('Do not alter predictions or test definitions to force historical values.')
